# Module 3 - Backprop From Scratch

## What you'll build

In Module 2 you derived a gradient by hand. That's fine for one neuron,
but real models have millions of parameters -- nobody derives those
gradients by hand. Instead we build an **autograd engine**: a small
system that records every arithmetic operation as it happens, then
replays them backwards to compute all the gradients automatically.
This is exactly what PyTorch does under the hood, just bigger.

You'll write a `Value` class. Each `Value` wraps a single number and
remembers:

* its `data` (the forward number),
* its `grad` (how much the final output changes if you nudge this
  number -- starts at 0),
* which other `Value`s produced it, and a local `_backward` function
  that knows how to push gradient to those producers.

When you write `c = a * b`, the engine creates `c` and stashes a closure
that, when called, adds `b.data * c.grad` to `a.grad` and `a.data *
c.grad` to `b.grad` -- the product rule. Calling `.backward()` on the
final node sorts the whole graph topologically and runs every local
`_backward` in reverse. That's backpropagation.

Implement `+`, `*`, `**`, the helpers built on them (`-`, `/`), the
`relu`/`tanh` nonlinearities, and `backward()`.

In [ ]:
import math

class Value:
    """A single scalar with a gradient and a recorded backward function."""

    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        # TODO: return a Value for self + other, and set its _backward so
        #       that d(out)/d(self) = 1 and d(out)/d(other) = 1.
        raise NotImplementedError

    def __mul__(self, other):
        # TODO: product rule. d(a*b)/da = b, d(a*b)/db = a.
        raise NotImplementedError

    def __pow__(self, other):
        # TODO: only int/float exponents. d(a**n)/da = n * a**(n-1).
        raise NotImplementedError

    def relu(self):
        # TODO: max(0, x); gradient flows only where the input was positive.
        raise NotImplementedError

    def tanh(self):
        # TODO: tanh(x); d(tanh)/dx = 1 - tanh(x)**2.
        raise NotImplementedError

    def backward(self):
        # TODO: topologically sort the graph, seed self.grad = 1.0, then
        #       run each node's _backward in reverse order.
        raise NotImplementedError

    # These are provided for you, built on the ops above.
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other): return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1
    def __repr__(self): return f'Value(data={self.data}, grad={self.grad})'


## Reveal the reference solution

Run the hidden cell to load the reference `Value` engine. This
**redefines** `Value` in the notebook with the complete implementation,
so the check below grades a working engine.

In [ ]:
# Reference solution (single source of truth: reference/autograd/engine.py)

"""A tiny scalar autograd engine (micrograd-style).

Each ``Value`` holds a single number and remembers how it was produced so
that we can replay the chain rule backwards through the computation graph.
"""

import math


class Value:
    """A single scalar with a gradient and a recorded backward function."""

    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)        # the forward value
        self.grad = 0.0                # d(output) / d(self), starts at zero
        self._backward = lambda: None  # how to push gradient to children
        self._prev = set(_children)    # the Values that produced this one
        self._op = _op                 # name of the op (for debugging)

    # ---- addition -----------------------------------------------------
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # d(a + b)/da = 1, d(a + b)/db = 1
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):  # other + self
        return self + other

    # ---- multiplication ----------------------------------------------
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # d(a * b)/da = b, d(a * b)/db = a
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other):  # other * self
        return self * other

    # ---- power (constant exponent) -----------------------------------
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float powers supported"
        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            # d(a**n)/da = n * a**(n-1)
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    # ---- negation and subtraction ------------------------------------
    def __neg__(self):  # -self
        return self * -1

    def __sub__(self, other):  # self - other
        return self + (-other)

    def __rsub__(self, other):  # other - self
        return other + (-self)

    # ---- division -----------------------------------------------------
    def __truediv__(self, other):  # self / other
        return self * other ** -1

    def __rtruediv__(self, other):  # other / self
        return other * self ** -1

    # ---- nonlinearities ----------------------------------------------
    def relu(self):
        """Rectified linear unit: max(0, x)."""
        out = Value(0.0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            # gradient flows through only where the input was positive
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        """Hyperbolic tangent."""
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            # d(tanh)/dx = 1 - tanh(x)**2
            self.grad += (1 - t * t) * out.grad
        out._backward = _backward
        return out

    # ---- backpropagation ---------------------------------------------
    def backward(self):
        """Run reverse-mode autodiff from this node back to the leaves."""
        # Build a topological ordering of all nodes in the graph via DFS.
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node._prev:
                    build_topo(child)
                topo.append(node)
        build_topo(self)

        # Seed the output gradient, then apply each local backward in reverse.
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


## Check your work

The grader runs two properties: a hand-checkable gradient is correct,
and gradients accumulate properly when a value is used more than once.

In [ ]:
import sys, os
# Make the repo root importable so `grader` / `reference` resolve.
_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)
from grader.checks.m3 import check_autograd
from grader.core import run_checks

r = run_checks(check_autograd(Value))
print('PASS' if r.passed else 'FAIL', f'score={r.score:.2f}')
if not r.passed:
    print('failed checks:', r.failed_checks)
